# 12 Methodology Update Summary

This notebook records the structural changes made in the repo update.

What changed:
- the active DAM ladder is now centralized as `FS0` through `FS4`
- the active model stack is seasonal naive baselines, `LEAR`, `XGBoost`, and `Prophet`
- `Prophet` is now an `FS2` entry only
- the first real shortlist point is after `FS2`, not after `FS1`
- the tuning cadence is now explicit by feature stage
- benchmark runners and notebooks are prepared, but not executed in this update


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


In [ ]:
summary_rows = [
    {"topic": "Active models", "decision": "naive_previous_day, naive_previous_week, naive_previous_year, LEAR, XGBoost, Prophet"},
    {"topic": "Prophet entry", "decision": "Prophet starts at FS2 once calendar and holiday structure is active"},
    {"topic": "Legacy removals", "decision": "ARIMA and SARIMA are removed from the active DAM stack and kept only as archived history"},
    {"topic": "FS ladder", "decision": "FS0 benchmarks, FS1 endogenous explicit, FS2 calendar/holiday plus Prophet, FS3 causal exogenous, FS4 advanced engineered finalists"},
    {"topic": "Shortlisting", "decision": "No shortlist after FS1; first fair shortlist after FS2"},
    {"topic": "Tuning cadence", "decision": "FS0 none, FS1 coarse, FS2 serious, FS3 mandatory retuning, FS4 selective finalist retuning"},
    {"topic": "Execution status", "decision": "No model runs were executed by this methodology update notebook refresh"},
]

display(pd.DataFrame(summary_rows))
display(feature_stage_policy_frame())
display(tuning_cadence_frame())
